# Data Mining — Identificación y consistencia química de metabolitos en modelos a escala genómica (GEMs)

Este notebook toma cualquier modelo COBRA en formato `.mat` (Recon3D, iDopaNeuro, el *core model* de E. coli, o cualquier otro con la misma estructura) y para cada metabolito:

1. Extrae su información declarada en el GEM (fórmula, carga, compartimento, nombre, en cuántas reacciones participa) — **Fase 2**.
2. Busca su InChI en cinco fuentes distintas (el propio GEM, PubChem, ChEBI, KEGG y HMDB), consultando por lotes y con caché en disco para que corra en minutos incluso en Recon3D — **Fase 3**.
3. Descompone cada InChI en sus capas químicas reales (fórmula, conectividad, hidrógenos, carga, estereoquímica) y calcula, por separado para cada capa, qué tan de acuerdo están las fuentes entre sí — **Fase 4**.
4. Compara lo que dice el GEM contra lo que dicen los InChIs, y resume los resultados en tablas y gráficas — **Fase 5**.

El notebook llega hasta ahí a propósito. Los dos pasos que siguen —agrupar los metabolitos por tipo de problema (Fase 6) y escoger/recomendar un único InChI "bueno" por metabolito (Fase 7)— necesitan las features de este notebook como insumo, así que quedan para después y se explican al final.

**Qué se mantiene del notebook original y qué cambia:** la idea de comparar varias fuentes y dar un puntaje proporcional (no todo-o-nada) es del diseño original y se conserva — de hecho se extiende a cada capa química por separado, no solo al InChI completo. Lo que cambia es *cómo* se separan las capas (antes se contaban los `/` del string, lo que no corresponde a capas químicas reales ni sirve para decidir si dos InChIs representan la misma molécula) y cómo se puntúa (antes "gana" una sola fuente; ahora cada fuente recibe su fracción de consenso). También se incorpora `pipeline_fase3.py` (el script que ya estaba en la carpeta `nuevo/`), que resolvió un problema real: las consultas a PubChem/ChEBI/KEGG en el notebook original eran una llamada web por metabolito, lo que hace im­práctico correr Recon3D (8,399 metabolitos). Aquí se consultan por lotes de 100 (PubChem/ChEBI) o 10 (KEGG), con caché y reintentos.

> **Para correr esto:** sube tus archivos `.mat` a Google Drive o directamente a la sesión de Colab, ajusta la celda de configuración (Fase 1.3) con los modelos que quieras procesar, y corre las celdas en orden. La Fase 3 necesita internet (consulta PubChem/ChEBI/KEGG) y RDKit (para convertir SMILES/.mol a InChI); fuera de Colab, si no tienes RDKit instalado, el notebook sigue funcionando para las Fases 2, 4 y 5 usando el InChI que cada modelo ya trae consigo (`metInChIString`), solo que sin los InChIs adicionales de las otras fuentes.


## Fase 1 — Preparación

Instalar dependencias, montar Drive (si aplica) y decir qué modelos procesar.


### 1.1 Dependencias

Se quitaron `pubchempy`, `libchebipy` y Open Babel del notebook original: `pipeline_fase3.py` ya no los necesita porque habla directo con las APIs REST de PubChem, ChEBI y KEGG (más rápido y sin el error 550 que daba libchebipy por FTP), y usa RDKit tanto para `.mol → InChI` como para `SMILES → InChI`. Corre esta celda una sola vez por sesión.


In [ ]:
!pip install -q rdkit requests scipy pandas openpyxl matplotlib
print("Dependencias instaladas.")

### 1.2 Montar Google Drive (opcional)

Si tus archivos `.mat` están en Drive, ejecuta esta celda. Si los subiste directo a la sesión de Colab (panel izquierdo → archivos), puedes saltarla.


In [ ]:
import os
if os.path.exists('/content'):
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("No estamos en Colab: se omite el montaje de Drive. Corriendo en VS Code / local, "
          "asegurate de que BASE_DIR (siguiente celda) apunte a la carpeta donde estan tus .mat.")

### 1.3 Configuración — qué modelos procesar

`MODELOS` es una lista de `(archivo, nombre_corto, limite)`. `limite=None` procesa todos los metabolitos; un número (p. ej. 200) sirve para probar rápido antes de correr el modelo completo. El notebook corre sobre **todos** los modelos de la lista en una sola pasada y al final junta los resultados en 3 tablas con una columna `Modelo` — así es como "funciona para cualquier modelo": basta con agregar o quitar entradas de esta lista, no hay que tocar el resto del notebook.


In [ ]:
import os

EN_COLAB = os.path.exists('/content')
BASE_DIR = ('/content/drive/MyDrive' if EN_COLAB and os.path.isdir('/content/drive/MyDrive')
            else '/content' if EN_COLAB else os.path.join(os.getcwd(), 'datos'))
OUT_DIR = os.path.join(BASE_DIR, 'resultados')
CACHE_PATH = os.path.join(OUT_DIR, 'cache_web.json')
DIR_MOL_HMDB = os.path.join(BASE_DIR, 'HMDBmols')   # .mol locales de HMDB (su sitio bloquea descarga automatica)
os.makedirs(OUT_DIR, exist_ok=True)

# Agrega o quita modelos aqui. El archivo debe estar en BASE_DIR.
MODELOS = [
    ('ecoli_core_model.mat', 'ecoli_core_model', None),
    ('iDopaNeuroC.mat',      'iDopaNeuroC',       None),
    ('Recon3D_301.mat',      'Recon3D_301',       None),
]

# Que fuentes externas consultar en la Fase 3 (ademas del InChI propio del GEM
# y del que sale de convertir el SMILES del GEM). Apagar una fuente aqui si no
# aplica (p. ej. no tienes carpeta de .mol de HMDB) en vez de que falle.
USAR_FUENTE = {'PUBCHEM': True, 'CHEBI': True, 'KEGG': True, 'HMDB': True}

print(f"BASE_DIR = {BASE_DIR}")
print(f"Modelos configurados: {[m[1] for m in MODELOS]}")

### 1.4 Imports


In [ ]:
import re, json, time, warnings
from collections import Counter
import numpy as np
import pandas as pd
import scipy.io
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

try:
    import requests
    HAY_REQUESTS = True
except ImportError:
    HAY_REQUESTS = False

try:
    from rdkit import Chem, DataStructs, RDLogger
    from rdkit.Chem import RDKFingerprint
    RDLogger.DisableLog('rdApp.*')
    HAY_RDKIT = True
except ImportError:
    HAY_RDKIT = False

if not HAY_RDKIT:
    print("Aviso: RDKit no esta disponible en este entorno. La Fase 3 (SMILES/.mol -> InChI "
          "y Tanimoto) se saltara; las Fases 2, 4 y 5 funcionan igual con el InChI propio del GEM.")
if not HAY_REQUESTS:
    print("Aviso: 'requests' no esta disponible. La Fase 3 no podra consultar PubChem/ChEBI/KEGG.")

## Fase 2 — Cargar cualquier modelo y extraer su información

El notebook original asumía que el struct de MATLAB se llamaba `model` y que los campos se llamaban exactamente `metNames`, `metFormulas`, etc. En la práctica cada modelo los nombra distinto: `Recon3D_301.mat` usa `metCharges`, `iDopaNeuroC.mat` usa `metCharge`; unos usan `metKEGGID`, otros `metKeggID`; `metInChIString` a veces aparece como `metInchiString`. Esto es lo que resolvió `pipeline_fase3.py` con un mapa de alias (`ALIASES`) y una función que detecta sola cuál variable del `.mat` es el modelo (busca la que tenga un campo `mets`), en vez de asumir que se llama `model`. Se retoma esa solución aquí tal cual, porque ya está probada contra los 3 modelos reales del proyecto.


In [ ]:
ALIASES = {
    'mets':           ['mets'],
    'metNames':       ['metNames'],
    'metFormulas':    ['metFormulas'],
    'metCharges':     ['metCharges', 'metCharge'],
    'metHMDBID':      ['metHMDBID', 'metHMDB'],
    'metKEGGID':      ['metKEGGID', 'metKeggID', 'metKeggId'],
    'metPubChemID':   ['metPubChemID', 'metPubChemId'],
    'metCHEBIID':     ['metCHEBIID', 'metChEBIID', 'metChebiID'],
    'metInChIString': ['metInChIString', 'metInchiString', 'metInChiString'],
    'metSmiles':      ['metSmiles', 'metSMILES'],
}


def cargar_modelo(ruta_mat):
    """Detecta automaticamente la variable del .mat que es el modelo (la que tiene 'mets'),
    en vez de asumir que se llama 'model' (Recon3D la llama 'Recon3D_301', iDopaNeuroC la
    llama 'iDopaNeuroC', etc.)."""
    mat = scipy.io.loadmat(ruta_mat)
    for clave in [k for k in mat if not k.startswith('__')]:
        try:
            if 'mets' in mat[clave].dtype.names:
                return mat[clave], clave
        except Exception:
            continue
    claves = [k for k in mat if not k.startswith('__')]
    return mat[claves[0]], claves[0]


def resolver_campo(variable, nombre_canonico):
    for nombre in ALIASES.get(nombre_canonico, [nombre_canonico]):
        if nombre in (variable.dtype.names or ()):
            return nombre
    return None


def extraer_alineado(variable, nombre_canonico, n_total):
    """Un valor por metabolito, en su posicion original (si falta, '' -- nunca se recorren
    posiciones, que fue un bug real del notebook original al saltarse vacios)."""
    campo_real = resolver_campo(variable, nombre_canonico)
    if campo_real is None:
        return [''] * n_total
    out = []
    for v in variable[campo_real][0, 0]:
        s = str(v[0][0]).strip() if (v.size > 0 and len(v[0]) > 0) else ''
        out.append('' if s in ('[]', 'nan', 'NaN') else s)
    return out


def extraer_charges(variable, n_total):
    campo_real = resolver_campo(variable, 'metCharges')
    if campo_real is None:
        return [None] * n_total
    out = []
    for v in variable[campo_real][0, 0]:
        try:
            out.append(int(v[0]) if v.size > 0 else None)
        except Exception:
            out.append(None)
    return out


def contar_reacciones(variable, n_total):
    """En cuantas reacciones participa cada metabolito: columnas no-cero por fila de la
    matriz estequiometrica S. Si el modelo no trae S, se devuelve None por metabolito en
    vez de fallar (algunos .mat de prueba no la incluyen)."""
    if 'S' not in (variable.dtype.names or ()):
        return [None] * n_total
    S = variable['S'][0, 0]
    try:
        return (S != 0).sum(axis=1).A1.tolist() if hasattr(S, 'A1') else (S != 0).sum(axis=1).tolist()
    except Exception:
        return [None] * n_total


_RE_COMPARTIMENTO = re.compile(r'_([a-zA-Z0-9]{1,2})$')


def compartimento_desde_id(met_id):
    """Convencion COBRA: el ID del metabolito termina en '_<compartimento>' (p. ej.
    'atp_c' = citosol, 'glc_e' = extracelular). Si no calza el patron, None."""
    m = _RE_COMPARTIMENTO.search(met_id) if met_id else None
    return m.group(1) if m else None

### Normalización de identificadores externos

Antes de poder *consultar* PubChem/ChEBI/KEGG/HMDB hay que limpiar lo que trae el GEM: algunos campos tienen varios IDs separados por `;` (se usa el primero), el ID de PubChem a veces viene como float (`"444.0"`) o con el prefijo `CID:`, y el de HMDB puede venir en formato viejo de 5 dígitos (`HMDB02111`) en vez del formato actual de 7 (`HMDB0002111`). Esto también viene de `pipeline_fase3.py`.


In [ ]:
def primer_id(s):
    partes = re.split(r'[;,|\s]+', s.strip()) if s else []
    return partes[0] if partes and partes[0] else ''

def norm_pubchem(s):
    s = primer_id(s).replace('CID:', '')
    s = s[:-2] if s.endswith('.0') else s
    return s if s.isdigit() else ''

def norm_chebi(s):
    s = primer_id(s).upper().replace('CHEBI:', '')
    return s if s.isdigit() else ''

def norm_kegg(s):
    s = primer_id(s).upper()
    return s if re.fullmatch(r'[CDG]\d{5}', s) else ''

def norm_hmdb(s):
    s = primer_id(s).upper()
    m = re.fullmatch(r'HMDB(\d+)', s)
    return f'HMDB{int(m.group(1)):07d}' if m else ''


## Fase 3 — Buscar el InChI de cada metabolito en varias fuentes

El notebook original pedía el InChI **de uno en uno** (una llamada web por metabolito, por fuente). Con 72 metabolitos del *core model* de E. coli eso es lento pero tolerable; con los 8,399 de Recon3D es impráctico. Esto es justo lo que resolvió `pipeline_fase3.py`, que ya estaba en la carpeta `nuevo/` y se incorpora aquí casi sin cambios porque ya está probado:

- **PubChem**: hasta 100 CIDs por llamada (`PUG-REST`, `property/InChI/JSON`).
- **ChEBI**: hasta 100 IDs por llamada, contra la API REST nueva (`chebi/backend/api/public`) — la librería `libchebipy` del notebook original usaba FTP y fallaba en Colab con error 550.
- **KEGG**: hasta 10 IDs por llamada (descarga el `.mol` y RDKit lo convierte a InChI); si algún ID no existe, KEGG lo omite sin avisar y el lote se repite de uno en uno para no desalinear los resultados.
- **HMDB**: su sitio bloquea descargas automáticas (Cloudflare). Si tienes una carpeta local con los `.mol` de HMDB (`DIR_MOL_HMDB`, un archivo por ID), se usan esos; si no, esta fuente queda vacía para ese modelo en vez de fallar.
- Todo se guarda en una **caché en disco** (`cache_web.json`): si se corre el notebook de nuevo, no se repiten consultas ya resueltas.
- Reintentos con espera creciente ante respuestas `429`/`503` (límite de velocidad de la API).


In [ ]:
def cargar_cache():
    if os.path.exists(CACHE_PATH):
        with open(CACHE_PATH) as f:
            return json.load(f)
    return {'PUBCHEM': {}, 'CHEBI': {}, 'KEGG': {}}

def guardar_cache(cache):
    with open(CACHE_PATH, 'w') as f:
        json.dump(cache, f)

def _get(url, **kw):
    for intento in range(4):
        try:
            r = requests.get(url, timeout=60, **kw)
            if r.status_code in (429, 503):
                time.sleep(2 * (intento + 1)); continue
            return r
        except requests.RequestException:
            time.sleep(2 * (intento + 1))
    return None

In [ ]:
def inchis_pubchem(cids, cache):
    """PubChem PUG-REST: hasta 100 CIDs por llamada."""
    faltan = sorted({c for c in cids if c and c not in cache})
    for i in range(0, len(faltan), 100):
        lote = faltan[i:i + 100]
        r = requests.post('https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/property/InChI/JSON',
                          data={'cid': ','.join(lote)}, timeout=120)
        encontrados = {}
        if r.ok:
            for p in r.json().get('PropertyTable', {}).get('Properties', []):
                encontrados[str(p['CID'])] = p.get('InChI')
        elif r.status_code != 404:
            print(f"  PubChem HTTP {r.status_code} en lote {i // 100}; se reintentara en la proxima corrida")
            continue
        for c in lote:
            cache[c] = encontrados.get(c)
        time.sleep(0.25)
    return cache


def inchis_chebi(ids, cache):
    """API REST nueva de ChEBI (ya no FTP via libchebipy)."""
    faltan = sorted({c for c in ids if c and c not in cache})
    for i in range(0, len(faltan), 100):
        lote = faltan[i:i + 100]
        r = _get('https://www.ebi.ac.uk/chebi/backend/api/public/compounds/',
                 params={'chebi_ids': ','.join(lote)})
        if r is None or not r.ok:
            print(f"  ChEBI fallo en lote {i // 100}; se reintentara en la proxima corrida")
            continue
        datos = r.json()
        for c in lote:
            d = (datos.get(c) or {}).get('data') or {}
            cache[c] = (d.get('default_structure') or {}).get('standard_inchi') or None
    return cache


def _mol_a_inchi(bloque):
    try:
        mol = Chem.MolFromMolBlock(bloque, sanitize=True)
        return Chem.MolToInchi(mol) or None if mol is not None else None
    except Exception:
        return None


def inchis_kegg(ids, cache):
    faltan = sorted({c for c in ids if c and c not in cache})
    for i in range(0, len(faltan), 10):
        lote = faltan[i:i + 10]
        r = _get('https://rest.kegg.jp/get/' + '+'.join(lote) + '/mol')
        texto = r.text if r is not None and r.ok else ''
        bloques = [b for b in re.split(r'\$\$\$\$\r?\n', texto) if 'M  END' in b]
        if len(bloques) == len(lote):
            for c, b in zip(lote, bloques):
                cache[c] = _mol_a_inchi(b)
        else:
            for c in lote:
                r1 = _get(f'https://rest.kegg.jp/get/{c}/mol')
                cache[c] = _mol_a_inchi(r1.text) if (r1 is not None and r1.ok and 'M  END' in r1.text) else None
                time.sleep(0.35)
        time.sleep(0.35)
    return cache


def inchis_hmdb_local(ids):
    if not os.path.isdir(DIR_MOL_HMDB):
        return {}
    out = {}
    for c in set(ids):
        ruta = os.path.join(DIR_MOL_HMDB, f'{c}.mol')
        if c and os.path.exists(ruta):
            with open(ruta) as f:
                out[c] = _mol_a_inchi(f.read())
    return out


def smiles_a_inchi(smiles):
    if not smiles:
        return None
    try:
        mol = Chem.MolFromSmiles(smiles)
        return Chem.MolToInchi(mol) or None if mol is not None else None
    except Exception:
        return None

### Correr la resolución sobre todos los modelos configurados

Para cada modelo de `MODELOS`: carga el `.mat`, extrae la información del GEM, normaliza los IDs externos, y para cada fuente activada en `USAR_FUENTE` obtiene el InChI correspondiente (en lote, con caché). El resultado es una tabla por modelo con una columna de InChI por fuente (`INCHI_MODELO`, `INCHI_SMILES`, `INCHI_PUBCHEM`, `INCHI_CHEBI`, `INCHI_KEGG`, `INCHI_HMDB`), que es exactamente el insumo que necesita la Fase 4. Si RDKit no está disponible, se usa únicamente `INCHI_MODELO` (el que el GEM ya trae) — las fases siguientes funcionan igual, solo que sin comparar contra fuentes externas.


In [ ]:
cache = cargar_cache() if HAY_REQUESTS else {'PUBCHEM': {}, 'CHEBI': {}, 'KEGG': {}}
tablas1 = {}   # nombre_modelo -> DataFrame (Tabla 1: extraccion + InChIs por fuente)

for archivo, nombre, limite in MODELOS:
    t0 = time.time()
    ruta = os.path.join(BASE_DIR, archivo)
    if not os.path.exists(ruta):
        print(f"Saltando {nombre}: no se encontro {ruta}")
        continue

    variable, var_key = cargar_modelo(ruta)
    n_total = variable[resolver_campo(variable, 'mets')][0, 0].shape[0]
    n = min(limite, n_total) if limite else n_total
    print(f"\n{'=' * 90}\n{nombre}: variable '{var_key}', {n_total} metabolitos (procesando {n})\n{'=' * 90}")

    col = lambda c: extraer_alineado(variable, c, n_total)[:n]
    t1 = pd.DataFrame({
        'Modelo': nombre, 'ID': col('mets'), 'Nombre': col('metNames'),
        'Formula': col('metFormulas'), 'Charge': extraer_charges(variable, n_total)[:n],
        'Compartimento': [compartimento_desde_id(x) for x in col('mets')],
        'Reaction_count': contar_reacciones(variable, n_total)[:n],
        'HMDB_ID': [norm_hmdb(x) for x in col('metHMDBID')],
        'KEGG_ID': [norm_kegg(x) for x in col('metKEGGID')],
        'PubChem_ID': [norm_pubchem(x) for x in col('metPubChemID')],
        'ChEBI_ID': [norm_chebi(x) for x in col('metCHEBIID')],
        'SMILES': col('metSmiles'),
    })
    t1['INCHI_MODELO'] = [x or None for x in col('metInChIString')]
    fuentes = ['INCHI_MODELO']

    if HAY_RDKIT:
        t1['INCHI_SMILES'] = t1['SMILES'].apply(smiles_a_inchi)
        fuentes.append('INCHI_SMILES')
        if HAY_REQUESTS and USAR_FUENTE.get('PUBCHEM'):
            inchis_pubchem(t1['PubChem_ID'], cache['PUBCHEM']); guardar_cache(cache)
            t1['INCHI_PUBCHEM'] = t1['PubChem_ID'].map(lambda c: cache['PUBCHEM'].get(c) or None if c else None)
            fuentes.append('INCHI_PUBCHEM')
        if HAY_REQUESTS and USAR_FUENTE.get('CHEBI'):
            inchis_chebi(t1['ChEBI_ID'], cache['CHEBI']); guardar_cache(cache)
            t1['INCHI_CHEBI'] = t1['ChEBI_ID'].map(lambda c: cache['CHEBI'].get(c) or None if c else None)
            fuentes.append('INCHI_CHEBI')
        if HAY_REQUESTS and USAR_FUENTE.get('KEGG'):
            inchis_kegg(t1['KEGG_ID'], cache['KEGG']); guardar_cache(cache)
            t1['INCHI_KEGG'] = t1['KEGG_ID'].map(lambda c: cache['KEGG'].get(c) or None if c else None)
            fuentes.append('INCHI_KEGG')
        if USAR_FUENTE.get('HMDB'):
            hmdb = inchis_hmdb_local(t1['HMDB_ID'])
            if hmdb:
                t1['INCHI_HMDB'] = t1['HMDB_ID'].map(hmdb.get)
                fuentes.append('INCHI_HMDB')

    t1.attrs['fuentes'] = fuentes
    tablas1[nombre] = t1
    cobertura = {f: int(t1[f].notna().sum()) if f in t1.columns else 0 for f in fuentes}
    print(f"Fuentes con InChI: {cobertura}  ({time.time() - t0:.0f} s)")

print("\nListo. Modelos procesados:", list(tablas1.keys()))

## Fase 4 — Descomponer cada InChI en sus capas reales y comparar por componente

Esta es la parte más importante del notebook (así quedó marcada en el plan de la tesis), así que vale la pena explicar el razonamiento completo.

**Qué estaba mal antes.** Tanto el notebook original como `pipeline_fase3.py` "contaban capas" haciendo `inchi.count('/')`. Eso cuenta caracteres, no capas químicas: no distingue cuáles de esas capas están presentes (fórmula, conectividad, hidrógenos, carga, estereoquímica...), y por lo tanto no sirve para decidir si dos InChIs representan la misma molécula o en qué se diferencian. Tampoco contaban el encabezado de versión (`InChI=1S`) como una capa — eso nunca estuvo mal — pero usar el conteo total como señal de calidad (`capas_comparison = 10 si count('/') == 7 else 0`) sí lo estaba: un InChI legítimo sin estereocentros tiene menos capas que uno con estereoquímica, y eso no lo hace "peor" ni "incompleto".

**Qué se hace ahora.** El InChI ya viene separado en capas con nombre (así lo define el *InChI Technical Manual*): `/`**fórmula**, `/c`**conectividad**, `/h`**hidrógenos**, `/q`+`/p`**carga**, `/b`+`/t`+`/m`+`/s`**estereoquímica**. Se separan esas capas por su prefijo (no por posición ni por conteo) y se comparan **una por una** entre las fuentes, en vez de comparar el InChI completo como un solo string. Esto es justo lo que pide el documento de metodología: en vez de decir "el InChI es diferente" (que no dice nada de *en qué* es diferente), se puede decir "la conectividad coincide en las 6 fuentes, pero la carga solo coincide en 4 de 6" — mucho más útil para encontrar el tipo de problema.

**Un hallazgo real al validar esto contra los 3 modelos del proyecto:** comparar la fórmula de la capa del InChI directamente contra la `Formula` del GEM solo coincidía en 20-34% de los casos. La razón es química, no un bug de extracción: la capa de fórmula del InChI describe la estructura de referencia *antes* de aplicar la capa de protones (`/p`), que es la que indica cuántos protones se agregaron o quitaron para llegar a la especie cargada que el GEM sí modela (p. ej. ATP⁴⁻ tiene `/p-4`: se quitaron 4 protones, por lo tanto 4 hidrógenos menos que en la fórmula de referencia). Ajustando el conteo de H por la capa de protones, la coincidencia sube a 88-97% en los 3 modelos — ese ajuste (`formula_real_desde_inchi`) es nuevo en este notebook, no estaba ni en el original ni en `pipeline_fase3.py`.

**Consenso proporcional, ahora por componente.** La idea original del estudiante se mantiene igual: si *k* de *N* fuentes coinciden en un valor, cada una de esas *k* recibe *k/N* (no todo-o-nada). Lo que cambia es que ahora se calcula **una vez por cada componente** (fórmula, conectividad, hidrógenos, carga, estereoquímica) y, además, en dos niveles agregados que pide el documento:

- **Estructura principal** = fórmula + conectividad + hidrógenos + carga (el documento la redefine así, incluyendo la carga).
- **InChI completo** = todo lo anterior + estereoquímica.

Con 6 fuentes: si las 6 coinciden → 1.0 cada una; 5 contra 1 → 0.833 / 0.167; 4 contra 2 → 0.667 / 0.333; 3 contra 3 → 0.5 / 0.5; las 6 distintas → 0.167 cada una. Una fuente sin InChI para ese metabolito no cuenta en el denominador ni recibe puntaje (no se le penaliza por no tener evidencia).

El Tanimoto de fingerprints (RDKit) se conserva como **feature secundaria** — mide similitud estructural global y es útil cuando dos InChIs no son idénticos pero sí muy parecidos (p. ej. un tautómero) — pero no reemplaza la comparación por capas, que es exacta y no depende de RDKit.


In [ ]:
"""
inchi_layers.py
================
Descomposición real de un InChI en sus capas químicas (según el InChI
Technical Manual) y cálculo de consenso proporcional entre fuentes.

No depende de RDKit ni de red: todo se obtiene con parsing de texto sobre
el propio string InChI, que ya trae la información codificada capa por
capa. Esto es intencional: así el notebook puede correr esta parte (la
más importante para la tesis, según el plan de Fase 4) incluso en un
entorno sin RDKit, y el resultado es idéntico a la interpretación química
real del InChI (no una aproximación).

Capas estándar de un InChI:
    InChI=<version>/<formula>/c<conectividad>/h<hidrógenos>/q<carga>/
          p<protones>/b<db-stereo>/t<tetrahedral-stereo>/m<parity>/
          s<stereo-type>/i<isótopos>/f<fixed-H>/r<reconectado-metales>

  - 'InChI=1S' (o 'InChI=1') es solo el encabezado de versión: NO es una
    capa química. Contar capas haciendo inchi.count('/') incluye este
    encabezado implícitamente y además no distingue qué capas están
    realmente presentes, por eso el notebook original (y pipeline_fase3.py)
    lo hacían mal con contar_capas_inchi = inchi.count('/').
  - La fórmula (primer segmento, sin letra de prefijo) sí es una capa.
  - El resto de capas se identifican por su letra de prefijo. Si una letra
    no aparece, esa capa no está presente (no es un error, p. ej. 'q'
    ausente significa carga neta 0).
"""

import re
from collections import Counter

PREFIJOS_CAPA = ('c', 'h', 'q', 'p', 'b', 't', 'm', 's', 'i', 'f', 'r')

_RE_FORMULA_ELEMENTO = re.compile(r'([A-Z][a-z]?)(\d*)')
_RE_CARGA_COMPONENTE = re.compile(r'^(\d+)\*([+-]?\d+)$')


def parse_inchi_layers(inchi):
    """Descompone un InChI en sus capas reales. Devuelve None si no es un
    InChI reconocible (p. ej. vacío, NaN, o no empieza con 'InChI=')."""
    if not isinstance(inchi, str) or not inchi.startswith('InChI='):
        return None
    cuerpo = inchi[len('InChI='):]
    partes = cuerpo.split('/')
    if not partes or not partes[0]:
        return None
    version = partes[0]                       # '1S' (estándar) o '1' (no estándar)
    capas = {p: None for p in PREFIJOS_CAPA}
    resto = partes[1:]
    # Caso especial: especies sin esqueleto, como el protón libre ('InChI=1S/p+1').
    # Ahi no hay capa de formula en absoluto: el primer segmento YA es una capa con
    # prefijo. Una formula real nunca empieza con una de las letras de prefijo en
    # minuscula (los simbolos de elemento siempre empiezan en mayuscula), asi que
    # ese caracter es la forma segura de distinguir los dos casos.
    if resto and resto[0] and resto[0][0] in capas:
        formula = None
    else:
        formula = resto[0] if resto else None
        resto = resto[1:]
    for segmento in resto:
        if not segmento:
            continue
        prefijo, valor = segmento[0], segmento[1:]
        if prefijo in capas:
            capas[prefijo] = valor
        # prefijos desconocidos (versiones futuras del estándar) se ignoran
        # sin romper el parseo del resto del InChI.
    n_capas_quimicas = (1 if formula else 0) + sum(1 for v in capas.values() if v is not None)
    return {
        'version': version,
        'es_estandar': version.endswith('S'),
        'formula': formula,
        'conectividad': capas['c'],
        'hidrogenos': capas['h'],
        'carga_q': capas['q'],
        'carga_p': capas['p'],
        'estereo_b': capas['b'],
        'estereo_t': capas['t'],
        'estereo_m': capas['m'],
        'estereo_s': capas['s'],
        'isotopos': capas['i'],
        'fixed_h': capas['f'],
        'reconectado': capas['r'],
        'n_capas_quimicas': n_capas_quimicas,
    }


def _suma_capa_carga(valor):
    """Suma los enteros con signo de una capa q o p. Soporta componentes
    múltiples separados por ';' (mezclas/sales) y la notación 'N*carga'
    que usa InChI para N componentes idénticos con la misma carga."""
    if not valor:
        return 0
    total = 0
    for parte in valor.split(';'):
        parte = parte.strip()
        if not parte:
            continue
        m = _RE_CARGA_COMPONENTE.match(parte)
        if m:
            total += int(m.group(1)) * int(m.group(2))
        else:
            try:
                total += int(parte)
            except ValueError:
                pass
    return total


def carga_desde_inchi(inchi):
    """Carga formal neta codificada en el propio InChI (capas q + p).
    Si ninguna de las dos capas está presente, el InChI no declara carga,
    lo cual en la convención de InChI significa carga neta 0 (neutro).
    Es exactamente la misma carga que calcularía RDKit a partir de la
    molécula, pero obtenida leyendo el string directamente."""
    capas = parse_inchi_layers(inchi)
    if capas is None:
        return None
    return _suma_capa_carga(capas['carga_q']) + _suma_capa_carga(capas['carga_p'])


def tiene_estereoquimica(inchi):
    capas = parse_inchi_layers(inchi)
    if capas is None:
        return False
    return any(capas[k] for k in ('estereo_b', 'estereo_t', 'estereo_m', 'estereo_s'))


def parse_formula(formula):
    """Fórmula molecular -> {elemento: cantidad}. Entero implícito = 1."""
    if not isinstance(formula, str) or not formula:
        return {}
    out = {}
    for elemento, cantidad in _RE_FORMULA_ELEMENTO.findall(formula):
        if not elemento:
            continue
        out[elemento] = out.get(elemento, 0) + (int(cantidad) if cantidad else 1)
    return out


def formulas_equal(f1, f2):
    """Compara dos fórmulas por composición elemental (no por string), así
    'C6H12O6' y 'H12C6O6' se reconocen como la misma fórmula."""
    if f1 is None or f2 is None:
        return False
    a, b = parse_formula(f1), parse_formula(f2)
    if not a or not b:
        return False
    return a == b


def formula_real_desde_inchi(inchi):
    """Fórmula de la especie realmente representada por el InChI, lista
    para compararse contra el 'Formula' que declara un GEM.

    Detalle que vale la pena explicar porque al principio produce muchos
    falsos 'no coincide': la capa de fórmula de un InChI (p. ej. 'C10H16N5
    O13P3' para ATP) describe la estructura de referencia ANTES de aplicar
    la capa de protones ('p'), no la especie cargada que normalmente
    modela un GEM. La capa 'p' indica cuántos protones se agregaron o
    quitaron para llegar a la especie real (p. ej. '/p-4' en ATP = se
    quitaron 4 protones), y cada protón quitado también quita un H de la
    fórmula. Ajustando el conteo de H por la capa p se obtiene la fórmula
    de la especie real -- la que sí es comparable con la del GEM. Validado
    contra los 3 modelos reales de este proyecto: sin este ajuste solo
    20-34% de las fórmulas coincidían con el GEM; con el ajuste, 88-97%.
    """
    capas = parse_inchi_layers(inchi)
    if capas is None or not capas['formula']:
        return None
    formula = parse_formula(capas['formula'])
    if capas['carga_p']:
        for parte in capas['carga_p'].split(';'):
            parte = parte.strip()
            if not parte:
                continue
            try:
                formula['H'] = formula.get('H', 0) + int(parte)
            except ValueError:
                pass
    return formula


# ---------------------------------------------------------------------
# Consenso proporcional (idea original del estudiante, aquí aplicada a
# cada componente por separado y no solo al InChI completo)
# ---------------------------------------------------------------------
def consenso_proporcional(valores):
    """valores: dict {fuente: valor_o_None}.
    Devuelve dict {fuente: score} donde score = (# fuentes con el mismo
    valor que esta fuente) / (# fuentes con algún valor). Las fuentes sin
    valor (None) no cuentan en el denominador y reciben score None (no se
    las penaliza por no tener evidencia; simplemente no participan).
    Ejemplos verificados (N=6 fuentes con valor):
      6 coinciden            -> 1.0 cada una
      5 coinciden, 1 distinta-> 0.833 / 0.167
      4 coinciden, 2 distinta-> 0.667 / 0.333
      3 y 3                  -> 0.5 / 0.5
      6 valores distintos    -> 0.167 cada una
    """
    presentes = {f: v for f, v in valores.items() if v is not None}
    n = len(presentes)
    if n == 0:
        return {f: None for f in valores}
    conteo = Counter(presentes.values())
    return {f: (conteo[presentes[f]] / n if f in presentes else None) for f in valores}


def componentes_inchi(inchi):
    """Para un InChI, devuelve los 5 componentes químicos nombrados en el
    documento de metodología: Formula, Connectivity, Hydrogens, Charge,
    Stereochemistry. 'Charge' se da como entero (no como string de capa)
    porque así dos representaciones equivalentes de la misma carga
    (p. ej. capa q ausente vs. capa p='0') se reconocen como iguales."""
    capas = parse_inchi_layers(inchi)
    if capas is None:
        return {'formula': None, 'conectividad': None, 'hidrogenos': None,
                 'carga': None, 'estereo': None}
    estereo = '|'.join(filter(None, (capas['estereo_b'], capas['estereo_t'],
                                      capas['estereo_m'], capas['estereo_s']))) or None
    return {
        'formula': capas['formula'],
        'conectividad': capas['conectividad'],
        'hidrogenos': capas['hidrogenos'],
        'carga': _suma_capa_carga(capas['carga_q']) + _suma_capa_carga(capas['carga_p']),
        'estereo': estereo,
    }


def consenso_por_componente(inchis_por_fuente):
    """inchis_por_fuente: dict {fuente: inchi_o_None}.
    Devuelve:
      - 'componentes': dict {fuente: {formula,conectividad,hidrogenos,carga,estereo}}
      - 'consenso': dict {componente: {fuente: score}} para cada uno de los 5
      - 'estructura_principal': consenso sobre (formula,conectividad,hidrogenos,carga)
      - 'inchi_completo': consenso sobre el InChI completo (string tal cual)
    """
    comp = {f: componentes_inchi(i) for f, i in inchis_por_fuente.items()}
    consenso = {}
    for nombre in ('formula', 'conectividad', 'hidrogenos', 'carga', 'estereo'):
        valores = {f: comp[f][nombre] for f in comp}
        consenso[nombre] = consenso_proporcional(valores)

    estructura_principal = {
        f: (comp[f]['formula'], comp[f]['conectividad'], comp[f]['hidrogenos'], comp[f]['carga'])
        if comp[f]['formula'] is not None else None
        for f in comp
    }
    inchi_completo = {f: inchis_por_fuente[f] for f in comp}

    return {
        'componentes': comp,
        'consenso': consenso,
        'estructura_principal': consenso_proporcional(estructura_principal),
        'inchi_completo': consenso_proporcional(inchi_completo),
    }


### Construir las 3 tablas finales

- **`metabolites`**: lo que el GEM declara de cada metabolito (uno por fila).
- **`identifiers`**: cada identificador externo, resuelto a su InChI, con el puntaje de consenso proporcional de esa fuente (una fila por metabolito × fuente). Aquí es donde vive el "¿qué tan de acuerdo está esta fuente con las demás?" a nivel de cada identificador.
- **`inchi_features`**: una fila por metabolito, con el consenso por componente (fórmula, conectividad, hidrógenos, carga, estereoquímica), los dos niveles agregados, la consistencia contra el GEM (Nivel 4 del documento de metodología) y el Tanimoto promedio como feature secundaria. Esta tabla es el insumo directo para la Fase 5 (y, más adelante, para la Fase 6 de clustering).


In [ ]:
def inchi_a_fingerprint(inchi):
    """Fingerprint RDKit a partir de un InChI, para el Tanimoto secundario de la Fase 4.
    Si RDKit no esta disponible esta funcion no se usa (ver HAY_RDKIT)."""
    if not inchi or not HAY_RDKIT:
        return None
    try:
        mol = Chem.MolFromInchi(inchi)
        return RDKFingerprint(mol) if mol is not None else None
    except Exception:
        return None


def val_o_none(x):
    """pandas convierte silenciosamente un None dentro de una columna de texto en
    NaN (float), y bool(NaN) es True -- un 'if valor:' o 'any(...)' sobre una celda
    asi cree que SI hay dato cuando en realidad esta vacia. Esta funcion normaliza
    cualquier variante de 'vacio' (None, NaN, '') a None de verdad, para que los
    chequeos de mas abajo funcionen como se espera."""
    if x is None or x == '':
        return None
    if isinstance(x, float) and pd.isna(x):
        return None
    return x


filas_metabolites, filas_identifiers, filas_features = [], [], []

for nombre, t1 in tablas1.items():
    fuentes = t1.attrs['fuentes']
    for _, fila in t1.iterrows():
        filas_metabolites.append({
            'Modelo': fila['Modelo'], 'ID': fila['ID'], 'Nombre': fila['Nombre'],
            'Formula': fila['Formula'], 'Charge': fila['Charge'],
            'Compartimento': fila['Compartimento'], 'Reaction_count': fila['Reaction_count'],
        })

        inchis_por_fuente = {f: val_o_none(fila.get(f)) for f in fuentes}
        if not any(inchis_por_fuente.values()):
            continue   # sin ningun InChI de ninguna fuente: no hay nada que comparar

        resultado = consenso_por_componente(inchis_por_fuente)

        for f in fuentes:
            ids_externos = {
                'INCHI_MODELO': fila['ID'], 'INCHI_SMILES': fila['SMILES'],
                'INCHI_PUBCHEM': fila.get('PubChem_ID'), 'INCHI_CHEBI': fila.get('ChEBI_ID'),
                'INCHI_KEGG': fila.get('KEGG_ID'), 'INCHI_HMDB': fila.get('HMDB_ID'),
            }
            if inchis_por_fuente[f] is None:
                continue
            filas_identifiers.append({
                'Modelo': fila['Modelo'], 'ID': fila['ID'],
                'Fuente': f.replace('INCHI_', ''), 'ID_externo': ids_externos.get(f),
                'InChI': inchis_por_fuente[f],
                'Score_EstructuraPrincipal': resultado['estructura_principal'][f],
                'Score_InChICompleto': resultado['inchi_completo'][f],
            })

        def _max_consenso(d):
            vals = [v for v in d.values() if v is not None]
            return max(vals) if vals else None

        inchi_modelo = inchis_por_fuente.get('INCHI_MODELO')
        f_real = formula_real_desde_inchi(inchi_modelo) if inchi_modelo else None
        carga_inchi = carga_desde_inchi(inchi_modelo) if inchi_modelo else None
        formula_match = (parse_formula(fila['Formula']) == f_real if (fila['Formula'] and f_real) else None)
        charge_match = (fila['Charge'] == carga_inchi
                         if (fila['Charge'] is not None and carga_inchi is not None) else None)
        capas_modelo = parse_inchi_layers(inchi_modelo) if inchi_modelo else None

        tanimoto_prom = None
        if HAY_RDKIT:
            fps = {f: inchi_a_fingerprint(inchis_por_fuente[f]) for f in fuentes if inchis_por_fuente[f]}
            fps = {f: fp for f, fp in fps.items() if fp is not None}
            if len(fps) >= 2:
                sims = []
                claves = list(fps)
                for i in range(len(claves)):
                    for j in range(i + 1, len(claves)):
                        sims.append(DataStructs.TanimotoSimilarity(fps[claves[i]], fps[claves[j]]))
                tanimoto_prom = round(float(np.mean(sims)), 3) if sims else None

        filas_features.append({
            'Modelo': fila['Modelo'], 'ID': fila['ID'], 'Nombre': fila['Nombre'],
            'SourceSupport': sum(1 for v in inchis_por_fuente.values() if v),
            'FormulaConsensus': _max_consenso(resultado['consenso']['formula']),
            'ConnectivityConsensus': _max_consenso(resultado['consenso']['conectividad']),
            'HydrogenConsensus': _max_consenso(resultado['consenso']['hidrogenos']),
            'ChargeConsensus': _max_consenso(resultado['consenso']['carga']),
            'StereoConsensus': _max_consenso(resultado['consenso']['estereo']),
            'EstructuraPrincipalConsensus': _max_consenso(resultado['estructura_principal']),
            'InChICompletoConsensus': _max_consenso(resultado['inchi_completo']),
            'FormulaMatch': formula_match,
            'ChargeMatch': charge_match,
            'EsEstandar': capas_modelo['es_estandar'] if capas_modelo else None,
            'TieneEstereo': tiene_estereoquimica(inchi_modelo) if inchi_modelo else None,
            'InChICompleteness': (round(sum([
                capas_modelo['conectividad'] is not None,
                capas_modelo['hidrogenos'] is not None,
                (capas_modelo['carga_q'] is not None or capas_modelo['carga_p'] is not None),
                any(capas_modelo[k] for k in ('estereo_b', 'estereo_t', 'estereo_m', 'estereo_s')),
            ]) / 4, 2) if capas_modelo else None),
            'Tanimoto_promedio': tanimoto_prom,
        })

metabolites = pd.DataFrame(filas_metabolites)
identifiers = pd.DataFrame(filas_identifiers)
inchi_features = pd.DataFrame(filas_features)

print(f"metabolites:    {len(metabolites)} filas")
print(f"identifiers:    {len(identifiers)} filas")
print(f"inchi_features: {len(inchi_features)} filas")
inchi_features.head()

## Fase 5 — Qué tan consistentes son los metabolitos, por modelo

Con `inchi_features` ya construida, el resto es responder preguntas concretas: ¿cuántos metabolitos tienen InChI? ¿En cuántos coincide la fórmula/carga del GEM con la del InChI? ¿Qué tipo de inconsistencia es más común? Esto es el Nivel 4 (GEM vs. InChI) y una vista agregada del Nivel 1-3 (consenso entre fuentes) del documento de metodología.


In [ ]:
resumen_filas = []
for nombre, t1 in tablas1.items():
    sub_feat = inchi_features[inchi_features['Modelo'] == nombre]
    n_total = len(t1)
    con_inchi = len(sub_feat)
    con_formula = sub_feat['FormulaMatch'].notna().sum()
    con_charge = sub_feat['ChargeMatch'].notna().sum()
    consistentes = ((sub_feat['FormulaMatch'] == True) & (sub_feat['ChargeMatch'] == True)).sum()
    solo_formula_mal = ((sub_feat['FormulaMatch'] == False) & (sub_feat['ChargeMatch'] != False)).sum()
    solo_charge_mal = ((sub_feat['ChargeMatch'] == False) & (sub_feat['FormulaMatch'] != False)).sum()
    ambos_mal = ((sub_feat['FormulaMatch'] == False) & (sub_feat['ChargeMatch'] == False)).sum()
    resumen_filas.append({
        'Modelo': nombre, 'Metabolitos': n_total, 'Con_InChI': con_inchi,
        'Cobertura_%': round(100 * con_inchi / n_total, 1) if n_total else None,
        'FormulaMatch_%': round(100 * sub_feat['FormulaMatch'].mean(), 1) if con_formula else None,
        'ChargeMatch_%': round(100 * sub_feat['ChargeMatch'].mean(), 1) if con_charge else None,
        'Consistentes': int(consistentes), 'Solo_formula_mal': int(solo_formula_mal),
        'Solo_charge_mal': int(solo_charge_mal), 'Formula_y_charge_mal': int(ambos_mal),
        'Con_estereo_%': round(100 * sub_feat['TieneEstereo'].mean(), 1) if con_inchi else None,
    })

resumen_modelos = pd.DataFrame(resumen_filas)
resumen_modelos

### Gráficas comparativas entre modelos


In [ ]:
PALETA = {'azul': '#2a78d6', 'naranja': '#eb6834', 'aqua': '#1baf7a',
          'amarillo': '#eda100', 'magenta': '#e87ba4', 'grid': '#e1e0d9', 'fondo': '#fcfcfb'}

def _estilo(ax):
    ax.set_facecolor(PALETA['fondo'])
    ax.grid(axis='y', color=PALETA['grid'], linewidth=0.8, zorder=0)
    ax.spines[['top', 'right']].set_visible(False)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.patch.set_facecolor(PALETA['fondo'])
modelos = resumen_modelos['Modelo']

ax = axes[0, 0]
ax.bar(modelos, resumen_modelos['Cobertura_%'], color=PALETA['azul'], zorder=3)
ax.set_title('Cobertura de InChI propio del GEM'); ax.set_ylabel('% de metabolitos'); _estilo(ax)

ax = axes[0, 1]
x = np.arange(len(modelos)); ancho = 0.35
ax.bar(x - ancho / 2, resumen_modelos['FormulaMatch_%'], ancho, label='Fórmula', color=PALETA['azul'], zorder=3)
ax.bar(x + ancho / 2, resumen_modelos['ChargeMatch_%'], ancho, label='Carga', color=PALETA['naranja'], zorder=3)
ax.set_xticks(x); ax.set_xticklabels(modelos); ax.set_title('GEM vs. InChI: % que coincide'); ax.legend(); _estilo(ax)

ax = axes[1, 0]
base = np.zeros(len(modelos))
for col, color, etiqueta in [('Consistentes', PALETA['aqua'], 'Consistente'),
                              ('Solo_formula_mal', PALETA['naranja'], 'Solo fórmula'),
                              ('Solo_charge_mal', PALETA['amarillo'], 'Solo carga'),
                              ('Formula_y_charge_mal', PALETA['magenta'], 'Ambas')]:
    valores = 100 * resumen_modelos[col] / resumen_modelos['Con_InChI']
    ax.bar(modelos, valores, bottom=base, label=etiqueta, color=color, zorder=3)
    base += valores.values
ax.set_title('Desglose de (in)consistencia GEM vs. InChI'); ax.set_ylabel('%'); ax.legend(fontsize=8); _estilo(ax)

ax = axes[1, 1]
datos_consenso = inchi_features['EstructuraPrincipalConsensus'].dropna()
ax.hist(datos_consenso, bins=10, color=PALETA['azul'], zorder=3, edgecolor='white')
ax.set_title('Consenso de "estructura principal" entre fuentes\n(1.0 = todas las fuentes de acuerdo)')
ax.set_xlabel('Score de consenso'); ax.set_ylabel('Metabolitos'); _estilo(ax)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'figuras_fase5.png'), dpi=150, facecolor=PALETA['fondo'])
plt.show()

### Validación manual rápida

Antes de confiar en las cifras agregadas, conviene revisar a mano un puñado de metabolitos muy conocidos (glucosa, ATP, piruvato...). Esta celda busca esos nombres en `metabolites` e imprime enlaces directos a KEGG/ChEBI/PubChem para compararlos a ojo — la misma idea de validación manual que tenía el notebook original (Fase 2 ahí), generalizada para que busque por nombre en cualquier modelo en vez de depender de `smallModel`.


In [ ]:
_LIGAS = {
    'KEGG':    lambda kid: f'https://www.genome.jp/dbget-bin/www_bget?{kid}',
    'ChEBI':   lambda cid: f'https://www.ebi.ac.uk/chebi/searchId.do?chebiId=CHEBI:{cid}',
    'PubChem': lambda pid: f'https://pubchem.ncbi.nlm.nih.gov/compound/{pid}',
}

def buscar_metabolitos_conocidos(nombres=('glucose', 'atp', 'pyruvate', 'nad', 'coa')):
    for nombre, t1 in tablas1.items():
        coincidencias = t1[t1['Nombre'].str.lower().str.contains('|'.join(nombres), na=False)]
        if coincidencias.empty:
            continue
        print(f"\n=== {nombre} ===")
        for _, fila in coincidencias.head(10).iterrows():
            print(f"- {fila['ID']} ({fila['Nombre']}): Formula GEM={fila['Formula']} Charge GEM={fila['Charge']}")
            if fila.get('KEGG_ID'):
                print(f"    KEGG:    {_LIGAS['KEGG'](fila['KEGG_ID'])}")
            if fila.get('ChEBI_ID'):
                print(f"    ChEBI:   {_LIGAS['ChEBI'](fila['ChEBI_ID'])}")
            if fila.get('PubChem_ID'):
                print(f"    PubChem: {_LIGAS['PubChem'](fila['PubChem_ID'])}")

buscar_metabolitos_conocidos()

### Exportar las 3 tablas


In [ ]:
OUTPUT_XLSX = os.path.join(OUT_DIR, 'resultados_datamining.xlsx')
with pd.ExcelWriter(OUTPUT_XLSX) as xl:
    metabolites.to_excel(xl, sheet_name='metabolites', index=False)
    identifiers.to_excel(xl, sheet_name='identifiers', index=False)
    inchi_features.to_excel(xl, sheet_name='inchi_features', index=False)
    resumen_modelos.to_excel(xl, sheet_name='resumen_por_modelo', index=False)

metabolites.to_csv(os.path.join(OUT_DIR, 'metabolites.csv'), index=False)
identifiers.to_csv(os.path.join(OUT_DIR, 'identifiers.csv'), index=False)
inchi_features.to_csv(os.path.join(OUT_DIR, 'inchi_features.csv'), index=False)

print(f"Guardado: {OUTPUT_XLSX}")
print(f"y también como CSV individuales en: {OUT_DIR}")

## Fases 6 y 7 — Agrupar metabolitos por tipo de problema y recomendar un InChI

Esta parte usa como insumo las tablas de la Fase 3 (`tablas1`, los InChIs por fuente) y revisa las variables de
la Fase 4 (`inchi_features`). Todo lo que sigue corre en menos de un minuto y guarda sus salidas en
`resultados/fase6_7/`: la tabla final de la Fase 7 en Excel y las figuras.

Primero van las funciones (capas del InChI, neutralización de la estructura, estereoquímica R/S, tipos de
inconsistencia, variables y regla de recomendación). Después, el análisis paso a paso. Requiere RDKit y
scikit-learn (Colab ya trae scikit-learn).

In [ ]:
"""Funciones de las Fases 6 y 7: capas del InChI, tipos de inconsistencia, variables para
clustering y recomendación del InChI. Pruebas en pruebas_inchi_calidad.py.

Idea central: el InChI estándar separa la estructura en capas. Comparar las fuentes capa por
capa dice QUÉ difiere (fórmula, conectividad, hidrógenos, estereoquímica o solo protonación),
no solo SI difiere.
"""
import re
from collections import Counter
from functools import lru_cache

import numpy as np
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import rdFingerprintGenerator

RDLogger.DisableLog("rdApp.*")

FUENTES = ["INCHI_MODELO", "INCHI_SMILES", "INCHI_PUBCHEM", "INCHI_CHEBI", "INCHI_KEGG", "INCHI_HMDB"]
DEL_MODELO = ["INCHI_MODELO", "INCHI_SMILES"]          # salen del mismo GEM: cuentan como un votante
EXTERNAS = ["INCHI_PUBCHEM", "INCHI_CHEBI", "INCHI_KEGG", "INCHI_HMDB"]   # HMDB solo si hay .mol locales

# Capas en orden de "más grave" a "menos grave". Cada nivel agrega capas al anterior.
NIVELES = [
    ("formula", ["formula"]),
    ("conectividad", ["c"]),
    ("hidrogenos", ["h"]),
    ("estereoquimica", ["b", "t", "m", "s", "i"]),
    ("carga", ["q", "p"]),
]


# ---------------------------------------------------------------- lectura de capas
@lru_cache(maxsize=None)
def a_estandar(inchi):
    """Devuelve el InChI estándar (InChI=1S/...). Los no estándar (InChI=1/) se convierten con RDKit."""
    if not isinstance(inchi, str) or not inchi.startswith("InChI="):
        return None
    inchi = inchi.strip()
    if inchi.startswith("InChI=1S/"):
        return inchi
    mol = Chem.MolFromInchi(inchi, sanitize=True)
    if mol is None:
        return None
    out = Chem.MolToInchi(mol) or None
    # /s2 = estereo relativa y /s3 = racémica: no dicen la configuración absoluta. RDKit la inventa al
    # convertir (/m0/s1), así que se quitan /t /m /s y el centro queda "no definido". Los E/Z (/b) se conservan.
    if out and re.search(r"/s[23](/|$)", inchi):
        partes = out.split("/")
        out = "/".join(partes[:2] + [x for x in partes[2:] if x[:1] not in ("t", "m", "s")])
    return out


def capas(inchi):
    """Diccionario capa -> texto. 'formula' es la capa principal sin prefijo."""
    if not isinstance(inchi, str) or not inchi.startswith("InChI="):
        return None
    partes = inchi.split("/")
    # El protón (InChI=1S/p+1) no tiene capa de fórmula: la segunda parte ya es una capa con letra.
    resto = partes[2:] if len(partes) > 1 and partes[1][:1].isupper() else partes[1:]
    d = {"formula": partes[1] if len(partes) > 1 and partes[1][:1].isupper() else ""}
    for p in resto:
        if p:
            d.setdefault(p[0], p[1:])      # primera aparición (las de /f fixed-H no se usan)
    return d


def clave(inchi, hasta):
    """Texto con las capas desde 'formula' hasta el nivel 'hasta' (inclusive)."""
    c = capas(inchi)
    if c is None:
        return None
    partes = []
    for nombre, letras in NIVELES:
        partes += [f"{l}={c.get(l, '')}" for l in letras]
        if nombre == hasta:
            break
    return "|".join(partes)


def n_capas(inchi):
    c = capas(inchi)
    return None if c is None else len(c)


# ---------------------------------------------------------------- carga y fórmula
def suma_capa_carga(valor):
    """Suma una capa /q o /p. Acepta varios componentes separados por ';' y la notación 'N*carga'
    (N componentes iguales con esa carga; p. ej. 'q2*-1;+2' = -2 + 2 = 0)."""
    total = 0
    for parte in (valor or "").split(";"):
        m = re.fullmatch(r"\s*(?:(\d+)\*)?([+-]?\d+)\s*", parte)
        if m:
            total += int(m.group(1) or 1) * int(m.group(2))
    return total


def carga_neta(inchi):
    """Carga neta = suma de /q (por componente) + protones de /p."""
    c = capas(inchi)
    if c is None:
        return None
    return suma_capa_carga(c.get("q")) + suma_capa_carga(c.get("p"))


_ELEM = re.compile(r"([A-Z][a-z]?)(\d*)")


def _elementos_validos():
    pt = Chem.GetPeriodicTable()
    out = set()
    for z in range(1, 119):
        out.add(pt.GetElementSymbol(z))
    return out


ELEMENTOS = _elementos_validos()


def parsear_formula(f):
    """'C2H3O2' -> {'C':2,'H':3,'O':2}. Fórmulas con varios componentes ('2C2H4O2.Na') se suman.
    Devuelve None si trae símbolos que no son elementos (R, X, FULLR...)."""
    if not isinstance(f, str) or not f.strip():
        return None
    total = Counter()
    for comp in f.strip().split("."):
        m = re.match(r"^(\d+)(.*)$", comp)
        mult, comp = (int(m.group(1)), m.group(2)) if m else (1, comp)
        if re.sub(r"[A-Z][a-z]?\d*", "", comp):
            return None
        for sim, n in _ELEM.findall(comp):
            if sim not in ELEMENTOS:
                return None
            total[sim] += mult * (int(n) if n else 1)
    return dict(total)


def hill(d):
    orden = (["C", "H"] if "C" in d else []) + sorted(k for k in d if not ("C" in d and k in ("C", "H")))
    return "".join(f"{k}{d[k] if d[k] != 1 else ''}" for k in orden if d.get(k, 0) > 0)


def _formula_de_capas(c):
    """Fórmula del InChI como diccionario. Vacía = cero átomos (el protón), no 'desconocida'."""
    return {} if c["formula"] == "" else parsear_formula(c["formula"])


def formula_corregida(inchi):
    """Fórmula del InChI ajustando los hidrógenos por la capa /p (la que reporta el GEM)."""
    c = capas(inchi)
    if c is None:
        return None
    d = _formula_de_capas(c)
    if d is None:
        return None
    d["H"] = d.get("H", 0) + suma_capa_carga(c.get("p"))
    return hill(d)


def coincide_formula_gem(inchi, formula_gem):
    """True/False si la fórmula del InChI (con su protonación) es la del GEM; None si no se puede comparar."""
    g = parsear_formula(formula_gem)
    f = formula_corregida(inchi)
    if g is None or f is None:
        return None
    return parsear_formula(f) == g


def coincide_pesados_gem(inchi, formula_gem):
    """True/False si coinciden los átomos distintos de H (no depende de la protonación); None si no se puede."""
    g = parsear_formula(formula_gem)
    c = capas(inchi)
    f = _formula_de_capas(c) if c else None
    if g is None or f is None:
        return None
    sin_h = lambda d: {k: v for k, v in d.items() if k != "H"}
    return sin_h(f) == sin_h(g)


def formula_generica(formula_gem):
    """True si la fórmula del GEM no es química (trae R, X, FULLR...): clases de lípidos, polímeros genéricos."""
    return isinstance(formula_gem, str) and bool(formula_gem.strip()) and parsear_formula(formula_gem) is None


# ---------------------------------------------------------------- estructura neutralizada y estereo
from rdkit.Chem import rdCIPLabeler
from rdkit.Chem.MolStandardize import rdMolStandardize

_UNCHARGER = rdMolStandardize.Uncharger()


def sin_carga(inchi):
    """El InChI sin la capa /p. La /q se conserva: es carga fija (p. ej. el N cuaternario de la colina)."""
    partes = inchi.split("/")
    return "/".join(partes[:2] + [p for p in partes[2:] if p[:1] != "p"])


@lru_cache(maxsize=None)
def _mol_neutra(inchi):
    """Molécula neutralizada con el Uncharger de RDKit (agrega o quita H donde se puede).
    Hace falta porque el InChI estándar no siempre marca la protonación con /p (p. ej. arsenito AsO3 q-3
    frente a ácido arsenioso AsH3O3)."""
    m = Chem.MolFromInchi(inchi)
    if m is None:
        return None
    try:
        return _UNCHARGER.uncharge(m)
    except Exception:
        return m


@lru_cache(maxsize=None)
def neutro(inchi):
    """InChI estándar de la estructura neutralizada."""
    m = _mol_neutra(inchi)
    return (Chem.MolToInchi(m) if m is not None else None) or sin_carga(inchi)


@lru_cache(maxsize=None)
def _estereo(inchi):
    """Configuraciones definidas: {('a', átomo): 'R'/'S', ('b', i, j): 'E'/'Z'} con el etiquetador CIP de RDKit.
    Los índices siguen la numeración canónica del InChI, así que son comparables entre fuentes que
    comparten la misma capa principal."""
    m = _mol_neutra(inchi)
    if m is None:
        return ()
    m = Chem.Mol(m)
    try:
        rdCIPLabeler.AssignCIPLabels(m)
    except Exception:
        return ()
    d = [(("a", a.GetIdx()), a.GetProp("_CIPCode")) for a in m.GetAtoms() if a.HasProp("_CIPCode")]
    d += [(("b", b.GetBeginAtomIdx(), b.GetEndAtomIdx()), b.GetProp("_CIPCode"))
          for b in m.GetBonds() if b.HasProp("_CIPCode")]
    return tuple(d)


def estereo(inchi):
    return dict(_estereo(inchi))


def principal(inchi):
    """Fórmula + conectividad + hidrógenos de la estructura neutralizada."""
    return clave(neutro(inchi), "hidrogenos")


def contradice(a, b):
    """Misma constitución y al menos un centro (o doble enlace) con configuración opuesta."""
    if principal(a) != principal(b):
        return False
    ea, eb = estereo(a), estereo(b)
    return any(ea[k] != eb[k] for k in set(ea) & set(eb))


def compatibles(a, b):
    """Misma constitución y ninguna configuración opuesta (una puede definir menos centros que la otra)."""
    return principal(a) == principal(b) and not contradice(a, b)


def especificidad(inchi):
    return len(_estereo(inchi))


def _estado_carga(inchi):
    c = capas(inchi)
    return (c.get("formula"), c.get("q", ""), c.get("p", "")) if c else None


# ---------------------------------------------------------------- inconsistencias
TIPOS = ["formula", "conectividad", "anillo_cadena", "hidrogenos", "estereo_contradictoria", "estereo_incompleta",
         "carga"]


@lru_cache(maxsize=None)
def n_anillos(inchi):
    m = _mol_neutra(inchi)
    return None if m is None else m.GetRingInfo().NumRings()


def _presentes(inchis):
    out = {}
    for f in FUENTES:
        v = a_estandar(inchis.get(f))
        if v:
            out[f] = v
    return out


def tipos_inconsistencia(inchis, normalizar=True):
    """Tipos presentes, en orden de gravedad. [] = todas coinciden. None = menos de 2 fuentes.
    Fórmula, conectividad e hidrógenos se comparan sobre la estructura neutralizada (capas anidadas: un tipo
    está si su capa separa fuentes que la anterior juntaba). La estereoquímica se compara con R/S y E/Z
    entre fuentes de igual constitución. 'carga' = misma constitución, distinto estado de protonación/carga."""
    p = _presentes(inchis) if normalizar else {k: v for k, v in inchis.items() if v}
    if len(p) < 2:
        return None
    tipos, previo = [], 1
    for nivel in ["formula", "conectividad", "hidrogenos"]:
        n = len({clave(neutro(v), nivel) for v in p.values()})
        if n > previo:
            tipos.append(nivel)
        previo = n
    if "conectividad" in tipos:
        # Misma fórmula, distinta conectividad: ¿es forma abierta vs cíclica (azúcares) o un isómero distinto?
        mismos = [(a, b) for i, a in enumerate(p.values()) for b in list(p.values())[i + 1:]
                  if clave(neutro(a), "formula") == clave(neutro(b), "formula")
                  and clave(neutro(a), "conectividad") != clave(neutro(b), "conectividad")]
        anillo = [n_anillos(a) != n_anillos(b) for a, b in mismos]
        if any(anillo):
            tipos.insert(tipos.index("conectividad") + 1, "anillo_cadena")
            if all(anillo):
                tipos.remove("conectividad")
    pares = [(a, b) for i, a in enumerate(p.values()) for b in list(p.values())[i + 1:] if principal(a) == principal(b)]
    if any(contradice(a, b) for a, b in pares):
        tipos.append("estereo_contradictoria")
    if any(not contradice(a, b) and set(estereo(a)) != set(estereo(b)) for a, b in pares):
        tipos.append("estereo_incompleta")
    if any(_estado_carga(a) != _estado_carga(b) for a, b in pares):
        tipos.append("carga")
    return tipos


# ---------------------------------------------------------------- consenso con compatibilidad
def _ganador(fuentes):
    """La estructura con más apoyo. Una fuente apoya a otra si son compatibles (misma constitución, sin
    configuraciones opuestas). Solo compiten las más específicas: una fuente que no define un centro no
    puede ganarle a una compatible que sí lo define. Devuelve (nombre, [nombres que la apoyan])."""
    nombres = list(fuentes)
    comp = {(a, b): compatibles(fuentes[a], fuentes[b]) for a in nombres for b in nombres}
    esp = {a: especificidad(fuentes[a]) for a in nombres}
    cand = [a for a in nombres if not any(comp[a, b] and esp[b] > esp[a] for b in nombres)]
    apoyo = {a: [b for b in nombres if comp[a, b]] for a in cand}
    mejor = max(cand, key=lambda a: (len(apoyo[a]), esp[a], -nombres.index(a)))
    return mejor, apoyo[mejor]


# ---------------------------------------------------------------- similitud
_GEN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)


@lru_cache(maxsize=None)
def _huella(inchi):
    m = _mol_neutra(inchi)
    return None if m is None else _GEN.GetFingerprint(m)


def tanimoto_minimo(inchis):
    fps = [fp for fp in (_huella(v) for v in inchis.values()) if fp is not None]
    if len(fps) < 2:
        return np.nan
    return float(min(min(DataStructs.BulkTanimotoSimilarity(fps[i], fps[i + 1:])) for i in range(len(fps) - 1)))


# ---------------------------------------------------------------- variables por metabolito
def _frac_moda(valores):
    if not valores:
        return np.nan
    return Counter(valores).most_common(1)[0][1] / len(valores)


def variables(inchis, formula_gem, carga_gem):
    """Variables de consistencia de un metabolito (las que lista el tutor)."""
    p = _presentes(inchis)
    fila = {
        "n_fuentes": len(p),
        "n_externas": sum(1 for f in EXTERNAS if f in p),
        "n_inchis_distintos": len(set(p.values())),
        "n_estructuras_distintas": len({neutro(v) for v in p.values()}),
        "consenso_inchi_completo": _frac_moda(list(p.values())),
        "consenso_capa_principal": _frac_moda([principal(v) for v in p.values()]),
        "consenso_estructura": (len(_ganador(p)[1]) / len(p)) if p else np.nan,
        "n_capas_media": float(np.mean([n_capas(v) for v in p.values()])) if p else np.nan,
        "tanimoto_minimo": tanimoto_minimo(p),
    }
    fg = [coincide_formula_gem(v, formula_gem) for v in p.values()]
    fg = [x for x in fg if x is not None]
    fila["frac_formula_gem"] = float(np.mean(fg)) if fg else np.nan
    fp = [coincide_pesados_gem(v, formula_gem) for v in p.values()]
    fp = [x for x in fp if x is not None]
    fila["frac_pesados_gem"] = float(np.mean(fp)) if fp else np.nan
    fila["formula_gem_generica"] = formula_generica(formula_gem)
    try:
        cg = int(carga_gem)
        fila["frac_carga_gem"] = float(np.mean([carga_neta(v) == cg for v in p.values()])) if p else np.nan
    except (TypeError, ValueError):
        fila["frac_carga_gem"] = np.nan
    tipos = tipos_inconsistencia(p, normalizar=False)
    for t in TIPOS:
        fila[f"inc_{t}"] = np.nan if tipos is None else int(t in tipos)
    fila["tipos_inconsistencia"] = "sin_comparacion" if tipos is None else (", ".join(tipos) or "ninguna")
    fila["tipo_principal"] = "sin_comparacion" if tipos is None else (tipos[0] if tipos else "consistente")
    interno = tipos_inconsistencia({k: p[k] for k in DEL_MODELO if k in p}, normalizar=False)
    fila["gem_interno"] = "sin_comparacion" if interno is None else (", ".join(interno) or "ninguna")
    return fila


# ---------------------------------------------------------------- recomendación (Fase 7)
def recomendar(inchis, carga_gem):
    """Propuesta de InChI recomendado. Reglas en fase6_7/PLAN.md (pendientes de aprobar por el tutor)."""
    p = _presentes(inchis)
    votos = {}
    gem = {k: p[k] for k in DEL_MODELO if k in p}
    if gem:   # el GEM vota una vez: su InChI, o el más específico de sus dos formas si son compatibles
        g = gem.get("INCHI_MODELO") or gem["INCHI_SMILES"]
        if len(gem) == 2 and compatibles(*gem.values()):
            g = max(gem.values(), key=especificidad)
        votos["GEM"] = g
    for f in EXTERNAS:
        if f in p:
            votos[f] = p[f]
    sal = {"inchi": None, "confianza": "revision_manual", "prioridad_revision": "baja",
           "motivo": "", "gem_coincide": None, "n_votantes": len(votos), "apoyo": 0, "fuentes_a_favor": ""}
    if not votos:
        sal["confianza"], sal["motivo"] = "sin_datos", "sin InChI en ninguna fuente"
        return sal
    mejor, a_favor = _ganador(votos)
    ext_a_favor = [k for k in a_favor if k != "GEM"]
    sal["apoyo"], sal["fuentes_a_favor"] = len(a_favor), ", ".join(k.replace("INCHI_", "") for k in a_favor)
    if len(votos) < 2:
        sal["motivo"] = "solo fuentes del modelo" if "GEM" in votos else "una sola base externa"
        return sal
    if len(a_favor) * 2 <= len(votos) or not ext_a_favor:
        sal["motivo"] = "fuentes en conflicto, sin mayoría"
        sal["prioridad_revision"] = "media"
        return sal
    if "GEM" in votos:
        sal["gem_coincide"] = "GEM" in a_favor
    elegido = votos[mejor]
    try:
        cg = int(carga_gem)
    except (TypeError, ValueError):
        cg = None
    misma = [v for v in p.values() if neutro(v) == neutro(elegido)]
    con_carga = [v for v in misma if cg is not None and carga_neta(v) == cg]
    sal["inchi"] = (con_carga or [elegido])[0]
    n_ext = sum(1 for k in votos if k != "GEM")
    sal["confianza"] = "alta" if (len(a_favor) == len(votos) and n_ext >= 2) else "media"
    if "GEM" in votos and "GEM" not in a_favor and len(ext_a_favor) >= 2:
        sal["prioridad_revision"] = "alta"
        sal["motivo"] = "la estructura del GEM difiere del consenso de las bases externas"
    elif len(a_favor) == len(votos):
        identicas = len({neutro(votos[k]) for k in a_favor}) == 1
        sal["motivo"] = "todas las fuentes coinciden" if identicas else \
            "coinciden; alguna fuente no define toda la estereoquímica"
    else:
        sal["motivo"] = "mayoría a favor"
    return sal


def base_id(met_id):
    """'atp[c]' -> 'atp' (quita el compartimento entre corchetes al final)."""
    return re.sub(r"\[[^\]]+\]$", "", str(met_id))

### Configuración

In [ ]:
import os, sys
import numpy as np
import pandas as pd

AQUI = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
RAIZ = os.path.dirname(AQUI) if os.path.basename(AQUI) == "fase6_7" else AQUI

# Dentro del notebook DataMining_GEM se usan las tablas de su Fase 3 (`tablas1`) y su carpeta de
# resultados (`OUT_DIR`). Fuera de él, se leen los Excel de la Fase 3 de `resultados/`.
EN_NOTEBOOK_TUTOR = "tablas1" in globals()
if EN_NOTEBOOK_TUTOR:
    SALIDA = os.path.join(OUT_DIR, "fase6_7")
    MODELOS_F67 = list(tablas1)
else:
    ENTRADA = os.path.join(RAIZ, "resultados")
    SALIDA = os.path.join(RAIZ, "resultados", "fase6_7")
    MODELOS_F67 = ["smallModel", "ecoli_core_model", "iDopaNeuroC", "Recon3D_301"]
os.makedirs(SALIDA, exist_ok=True)
SEMILLA = 42

### 1. Carga y fusión de compartimentos

In [ ]:
filas, conteo = [], []
for m in MODELOS_F67:
    if EN_NOTEBOOK_TUTOR:
        t1 = tablas1[m].copy()
    else:
        t1 = pd.read_excel(os.path.join(ENTRADA, f"{m}_tablas.xlsx"), "Tabla1_Extraccion")
    for f in FUENTES:
        if f not in t1.columns:
            t1[f] = None
    t1["Modelo"] = m
    t1["Metabolito"] = t1["ID"].astype(str).map(base_id)
    comps = t1.groupby("Metabolito")["ID"].apply(lambda s: ", ".join(s.astype(str)))
    u = t1.drop_duplicates("Metabolito").copy()
    u["Compartimentos"] = u["Metabolito"].map(comps)
    conteo.append({"Modelo": m, "Filas_Fase3": len(t1), "Metabolitos_unicos": len(u)})
    filas.append(u)
datos = pd.concat(filas, ignore_index=True)
conteo = pd.DataFrame(conteo)
print(conteo.to_string(index=False))

### 2. Variables de consistencia y recomendación por metabolito

In [ ]:
def por_fila(r):
    inchis = {f: r[f] for f in FUENTES}
    v = variables(inchis, r["Formula"], r["Charge"])
    rec = recomendar(inchis, r["Charge"])
    v.update({f"rec_{k}": x for k, x in rec.items()})
    return pd.Series(v)

var = datos.apply(por_fila, axis=1)
tabla = pd.concat([datos, var], axis=1)
print(tabla[["n_fuentes", "tipo_principal", "rec_confianza"]].describe(include="all").to_string())

## Fase 6.0 — ¿Qué variables de la Fase 4 sirven para agrupar?
Antes de agrupar se revisó qué miden las variables de `inchi_features`. Comparan cada capa del InChI como
texto, tal cual llega de cada fuente. Al revisar casos a mano aparecieron cuatro efectos que no son
diferencias químicas: (1) InChIs no estándar (`InChI=1/`) frente a estándar; (2) estereoquímica relativa
(`/s2`), que no fija la configuración absoluta; (3) la protonación, que cambia `/p` y a veces la fórmula
(p. ej. arsenito AsO3³⁻ frente a ácido arsenioso As(OH)3) sin cambiar la molécula; (4) una fuente sin capa
de estereo cuenta como "sin dato", así que el consenso de estereo no detecta estereoquímica incompleta.
La celda siguiente cuantifica cuánto pesa cada efecto. Por eso las Fases 6–7 calculan sus variables sobre
InChIs estándar, estructuras neutralizadas (RDKit) y configuraciones R/S, y conservan las de la Fase 4
como referencia.

### 3a. Cuánto pesan esos efectos en el "InChI completo distinto" de la Fase 4

In [ ]:
if "inchi_features" in globals():
    _efecto = []
    for _m, _t in tablas1.items():
        for _, _r in _t.iterrows():
            _p = {f: _r[f] for f in FUENTES if f in _t.columns and isinstance(_r[f], str) and _r[f].startswith("InChI=")}
            if len(_p) < 2 or len(set(_p.values())) == 1:
                continue
            _e = {f: a_estandar(v) for f, v in _p.items()}
            _e = {f: v for f, v in _e.items() if v}
            _efecto.append({"Modelo": _m,
                            "solo notacion (estandar vs no estandar)": len(set(_e.values())) <= 1,
                            "solo protonacion": len(set(_e.values())) > 1 and len({neutro(v) for v in _e.values()}) == 1,
                            "estructura realmente distinta": len({neutro(v) for v in _e.values()}) > 1})
    _efecto = pd.DataFrame(_efecto)
    tabla_efectos = _efecto.groupby("Modelo").sum().astype(int)
    tabla_efectos.loc["Total"] = tabla_efectos.sum()
    tabla_efectos.insert(0, "InChI completo distinto (Fase 4)", tabla_efectos.sum(axis=1))
    print("Filas (metabolito × compartimento) con ≥2 fuentes cuyo InChI completo no coincide tal cual:")
    print(tabla_efectos.to_string())

## Fase 6.1 — Análisis general
Se comparan las fuentes capa por capa del InChI. Un tipo de inconsistencia está presente si su
capa separa fuentes que la capa anterior todavía juntaba (fórmula → conectividad → hidrógenos →
estereoquímica → carga/protonación). Un metabolito puede tener más de un tipo.

### 3. Consistentes vs inconsistentes y tipos más frecuentes

In [ ]:
comparables = tabla[tabla["n_fuentes"] >= 2]
gen = []
for m in MODELOS_F67 + ["Total"]:
    t = tabla if m == "Total" else tabla[tabla["Modelo"] == m]
    c = t[t["n_fuentes"] >= 2]
    fila = {"Modelo": m, "Metabolitos": len(t), "Sin InChI": int((t["n_fuentes"] == 0).sum()),
            "Una fuente": int((t["n_fuentes"] == 1).sum()), "Comparables (≥2 fuentes)": len(c),
            "Consistentes (InChI idéntico)": int((c["tipo_principal"] == "consistente").sum()),
            "Con inconsistencias": int((c["tipo_principal"] != "consistente").sum())}
    for tp in TIPOS:
        fila[f"inc_{tp}"] = int(c[f"inc_{tp}"].sum())
    gen.append(fila)
general = pd.DataFrame(gen)
print(general.to_string(index=False))

### 4. ¿Qué fuente es la que discrepa? (frente a la estructura ganadora de cada metabolito)

In [ ]:
filas = []
for _, r in comparables[comparables["n_externas"] >= 2].iterrows():   # solo donde hay mayoría real
    p = {f: a_estandar(r[f]) for f in FUENTES}
    p = {f: v for f, v in p.items() if v}
    g = p[_ganador(p)[0]]
    for f, v in p.items():
        misma = principal(v) == principal(g)
        filas.append({"fuente": f.replace("INCHI_", ""),
                      "constitucion distinta": not misma,
                      "estereo contradictoria": misma and contradice(v, g),
                      "estereo menos definida": misma and not contradice(v, g) and especificidad(v) < especificidad(g),
                      "carga distinta a la del GEM": carga_neta(v) != r["Charge"]})
discrepa = (pd.DataFrame(filas).groupby("fuente").mean() * 100).round(1)
discrepa.columns = [f"% {c}" for c in discrepa.columns]
print(discrepa.to_string())

### 5. Distribución de los scores de consenso

In [ ]:
import matplotlib
try:
    get_ipython  # noqa: F821  (existe dentro de Jupyter/Colab)
    EN_JUPYTER = True
except NameError:
    EN_JUPYTER = False
    matplotlib.use("Agg")
import matplotlib.pyplot as plt


def cerrar(fig):
    """En el notebook muestra la figura; corriendo como script solo la cierra (ya quedó guardada)."""
    plt.show() if EN_JUPYTER else plt.close(fig)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, col, tit in [(ax[0], "consenso_inchi_completo", "InChI completo"),
                    (ax[1], "consenso_estructura", "Estructura (sin protonación)")]:
    for m in ["ecoli_core_model", "iDopaNeuroC", "Recon3D_301"]:
        a.hist(comparables.loc[comparables["Modelo"] == m, col], bins=np.linspace(0, 1, 11),
               alpha=0.5, label=m, density=True)
    a.set_title(f"Consenso: {tit}"); a.set_xlabel("fracción de fuentes que coinciden con la mayoría")
ax[0].set_ylabel("densidad"); ax[1].legend()
fig.tight_layout(); fig.savefig(os.path.join(SALIDA, "fig1_consenso.png"), dpi=150); cerrar(fig)

fig, ax = plt.subplots(figsize=(7, 4))
(general.set_index("Modelo").loc[MODELOS_F67, [f"inc_{t}" for t in TIPOS]]
 .div(general.set_index("Modelo").loc[MODELOS_F67, "Comparables (≥2 fuentes)"], axis=0) * 100
 ).rename(columns=lambda c: c[4:]).plot.bar(ax=ax)
ax.set_ylabel("% de metabolitos comparables"); ax.set_title("Tipos de inconsistencia por modelo")
fig.tight_layout(); fig.savefig(os.path.join(SALIDA, "fig2_tipos_inconsistencia.png"), dpi=150); cerrar(fig)

### 6. Casos representativos

In [ ]:
cols_caso = ["Modelo", "Metabolito", "Nombre", "Formula", "Charge", "n_fuentes", "tipos_inconsistencia",
             "consenso_estructura", "tanimoto_minimo", "rec_motivo"]
bien = comparables[(comparables["n_externas"] == 3) & (comparables["consenso_estructura"] == 1)
                   & (comparables["frac_carga_gem"] < 1)].head(5)
perfecto = comparables[(comparables["n_externas"] == 3) & (comparables["tipo_principal"] == "consistente")].head(5)
problema = tabla[tabla["rec_prioridad_revision"] == "alta"].sort_values("tanimoto_minimo").head(10)
print("Consistentes en todo:"); print(perfecto[cols_caso].to_string(index=False))
print("\nMisma estructura, solo cambia la protonación:"); print(bien[cols_caso].to_string(index=False))
print("\nProblemáticos (el GEM difiere del consenso externo):"); print(problema[cols_caso].to_string(index=False))

## Fase 6.2 — Clustering no supervisado
Unidad: metabolito con ≥2 fuentes (con una sola no hay consistencia que medir). Los que aparecen en
varios modelos con los mismos datos se cuentan una vez. Variables elegidas tras explorar (ver PLAN.md):
se quitaron las redundantes (|ρ| ≥ 0.9) y las que describen la molécula y no su anotación.

### 7. Variables y selección de k

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA

VARS = ["n_externas", "consenso_inchi_completo", "consenso_estructura", "consenso_capa_principal",
        "tanimoto_minimo", "frac_carga_gem", "frac_pesados_gem"]
llave = ["Metabolito", "Formula", "Charge"] + FUENTES
C = comparables[~comparables["formula_gem_generica"]].drop_duplicates(subset=llave).reset_index(drop=True)
print(f"Para clustering: {len(C)} metabolitos únicos (de {len(comparables)} filas comparables en {len(MODELOS_F67)} modelos; "
      f"{int(comparables['formula_gem_generica'].sum())} con fórmula genérica en el GEM van aparte)")
X = StandardScaler().fit_transform(C[VARS].astype(float))

sel = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=20, random_state=SEMILLA).fit(X)
    wa = AgglomerativeClustering(n_clusters=k, linkage="ward").fit(X)
    for nombre, lab in [("K-means", km.labels_), ("Jerárquico (Ward)", wa.labels_)]:
        sel.append({"metodo": nombre, "k": k, "silhouette": silhouette_score(X, lab, random_state=SEMILLA),
                    "davies_bouldin": davies_bouldin_score(X, lab)})
sel = pd.DataFrame(sel)
print(sel.pivot(index="k", columns="metodo", values=["silhouette", "davies_bouldin"]).round(3).to_string())

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for a, met in zip(ax, ["silhouette", "davies_bouldin"]):
    for nombre, d in sel.groupby("metodo"):
        a.plot(d["k"], d[met], marker="o", label=nombre)
    a.axvline(8, color="grey", ls="--", lw=0.8); a.set_xlabel("k"); a.set_title(met)
ax[0].legend(); fig.tight_layout(); fig.savefig(os.path.join(SALIDA, "fig3_seleccion_k.png"), dpi=150); cerrar(fig)

**Elección: K-means con k = 8.** k = 2 tiene el mejor silhouette pero solo separa "hay bases externas" de
"no hay". Entre k ≥ 3, k = 8 tiene el silhouette más alto y el Davies-Bouldin más bajo, y cada grupo tiene
una lectura química clara. Con k = 7 se juntan "consistente con bases externas" y "consistente solo con el
GEM", que no significan lo mismo. K-means supera a Ward en todos los k.

### 8. Grupos finales y su interpretación

In [ ]:
K = 8
print(sel[(sel["metodo"] == "K-means") & (sel["k"] == K)].round(3).to_string(index=False))
modelo_km = KMeans(n_clusters=K, n_init=20, random_state=SEMILLA).fit(X)
C["grupo_num"] = modelo_km.labels_
perfil = C.groupby("grupo_num")[VARS + [f"inc_{t}" for t in TIPOS] + ["n_capas_media"]].mean()
perfil.insert(0, "n", C["grupo_num"].value_counts().sort_index())

def nombrar(p):
    """Reglas sobre el promedio del grupo (orden = de lo más específico a lo más general)."""
    if p["frac_pesados_gem"] < 0.5:
        return "La fórmula del GEM no corresponde a su estructura"
    if p["consenso_capa_principal"] < 0.9 or p["tanimoto_minimo"] < 0.6:
        return "Discrepancias de constitución (fórmula, conectividad o anillo/cadena)"
    if p["inc_estereo_contradictoria"] > 0.5:
        return "Estereoquímica contradictoria (posible enantiómero equivocado)"
    if p["frac_carga_gem"] < 0.2:
        return "Carga/fórmula del GEM ionizadas, estructura neutra"
    if p["inc_estereo_incompleta"] > 0.5:
        return "Estereoquímica incompleta, poca evidencia externa"
    if p["n_externas"] < 1:
        return "Información insuficiente (solo el GEM)"
    if p["frac_carga_gem"] < 0.7:
        return "Misma estructura, difiere solo la protonación"
    return "Alta consistencia"

perfil.insert(0, "grupo", perfil.apply(nombrar, axis=1))
if not perfil["grupo"].is_unique:   # con otra versión de RDKit podría pasar: se numeran en vez de detenerse
    print("Aviso: dos grupos recibieron el mismo nombre; se numeran. Revisar el perfil antes de interpretar.")
    perfil["grupo"] = [f"{g} ({i + 1})" if (perfil["grupo"] == g).sum() > 1 else g
                       for i, g in enumerate(perfil["grupo"])]
C["grupo"] = C["grupo_num"].map(perfil["grupo"])
print(perfil.round(2).to_string())

coords = PCA(n_components=2, random_state=SEMILLA).fit(X)
P = coords.transform(X)
fig, ax = plt.subplots(figsize=(8, 6))
rng = np.random.default_rng(SEMILLA)
for g, d in C.groupby("grupo"):
    i = d.index.values
    ax.scatter(P[i, 0] + rng.normal(0, 0.05, len(i)), P[i, 1] + rng.normal(0, 0.05, len(i)), s=8, alpha=0.6,
               label=f"{g} (n={len(i)})")
v = coords.explained_variance_ratio_ * 100
ax.set_xlabel(f"PC1 ({v[0]:.0f} %)"); ax.set_ylabel(f"PC2 ({v[1]:.0f} %)")
ax.set_title(f"Grupos de consistencia estructural (K-means, k = {K})")
ax.legend(fontsize=7, loc="best"); fig.tight_layout()
fig.savefig(os.path.join(SALIDA, "fig4_grupos_pca.png"), dpi=150); cerrar(fig)

hm = perfil.set_index("grupo")[VARS]
fig, ax = plt.subplots(figsize=(9, 4))
norm = (hm - hm.min()) / (hm.max() - hm.min()).replace(0, 1)
im = ax.imshow(norm, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(VARS))); ax.set_xticklabels(VARS, rotation=35, ha="right", fontsize=8)
ax.set_yticks(range(len(hm))); ax.set_yticklabels(hm.index, fontsize=8)
for (r, c_), val in np.ndenumerate(hm.values):
    ax.text(c_, r, f"{val:.2f}", ha="center", va="center", fontsize=7,
            color="black" if norm.values[r, c_] > 0.6 else "white")
ax.set_title("Perfil promedio de cada grupo (color escalado por columna)")
fig.tight_layout(); fig.savefig(os.path.join(SALIDA, "fig5_perfil_grupos.png"), dpi=150); cerrar(fig)

### Interpretación química de los grupos
Los nombres salen de reglas fijas sobre el promedio de cada grupo (función `nombrar`), así que la lectura
es la misma en cada corrida. La tabla de perfiles y la figura 5 muestran los números en que se apoya.

- **Alta consistencia.** Varias bases externas y todas coinciden, incluso en la carga. Suelen ser
  moléculas pequeñas, neutras y sin grupos ionizables (agua, CO2, etanol). Anotación correcta.
- **Misma estructura, difiere solo la protonación.** Varias bases externas; al neutralizar, la estructura
  es idéntica, pero el GEM la guarda a pH fisiológico (fosfatos y carboxilatos desprotonados: NAD, CoA,
  PEP) y las bases en forma neutra. No es error de anotación: es el estado químico que el modelo necesita
  para balancear cargas en las reacciones. Explica la baja coincidencia de carga de la Fase 3.
- **Estereoquímica contradictoria.** Alguna fuente da la configuración opuesta (R frente a S) en al menos
  un centro. Es el problema más serio: el GEM o una base puede estar apuntando al enantiómero equivocado
  (p. ej. `lac_D` en ecoli: su InChI es L-lactato).
- **Discrepancias de constitución.** Fórmula o conectividad distintas: azúcares en forma abierta en una
  fuente y cíclica en otra (mismo metabolito), poliglutamatos o polímeros representados con otro tamaño, e
  IDs externos que apuntan a otro compuesto (p. ej. galactosamina frente a galactosamina-6-fosfato).
- **Carga/fórmula del GEM ionizadas, estructura neutra.** Ninguna fuente, ni el InChI del propio GEM, tiene
  la carga declarada en el GEM: los campos de fórmula y carga del modelo no corresponden a su estructura.
  Es una inconsistencia interna del GEM, casi toda de Recon3D.
- **La fórmula del GEM no corresponde a su estructura.** Los átomos pesados de la fórmula del GEM no son
  los del InChI (p. ej. una acilcarnitina C20:4 con InChI de C24). Error de anotación en el GEM.
- **Estereoquímica incompleta, poca evidencia externa.** El InChI del GEM trae estereoquímica relativa
  (`/s2`) o no define centros que su SMILES sí define, y casi no hay bases externas para confirmar.
- **Información insuficiente (solo el GEM).** El GEM es consistente consigo mismo, pero no hay bases
  externas que lo validen; no se puede afirmar que la anotación sea correcta ni incorrecta.

## Fase 7 — Tabla final por metabolito

### 9. Grupo, estado de la anotación y tabla final

In [ ]:
tabla = tabla.merge(C[llave + ["grupo"]], on=llave, how="left")
tabla.loc[tabla["n_fuentes"] == 0, "grupo"] = "Sin InChI en ninguna fuente (regla)"
tabla.loc[tabla["n_fuentes"] == 1, "grupo"] = "Una sola fuente (regla)"
tabla.loc[(tabla["n_fuentes"] >= 2) & tabla["formula_gem_generica"], "grupo"] = "Fórmula genérica en el GEM (regla)"
assert tabla["grupo"].notna().all()

def estado(r):
    if r["rec_confianza"] == "sin_datos" or r["rec_motivo"] in ("solo fuentes del modelo", "una sola base externa"):
        return "Sin evidencia externa suficiente"
    if r["rec_confianza"] == "revision_manual":
        return "Inconsistente: fuentes en conflicto"
    if r["rec_gem_coincide"] is False:
        return "Inconsistente: el GEM difiere del consenso externo"
    try:
        carga_ok = carga_neta(r["rec_inchi"]) == int(r["Charge"])
    except (TypeError, ValueError):
        carga_ok = False
    pesados_ok = coincide_pesados_gem(r["rec_inchi"], r["Formula"]) is not False
    if not (carga_ok and pesados_ok):
        return "Inconsistente: carga/fórmula del GEM no coinciden con la estructura"
    return "Correctamente anotado"

tabla["Estado_anotacion"] = tabla.apply(estado, axis=1)
tabla["Fuentes"] = tabla.apply(lambda r: ", ".join(f[6:] for f in FUENTES if isinstance(r[f], str) and r[f].startswith("InChI")), axis=1)

final = tabla.rename(columns={
    "Formula": "Formula_GEM", "Charge": "Carga_GEM", "n_fuentes": "N_fuentes",
    "n_inchis_distintos": "N_InChIs_distintos", "consenso_inchi_completo": "Consenso_InChI_completo",
    "consenso_estructura": "Consenso_estructura", "tanimoto_minimo": "Tanimoto_minimo",
    "tipos_inconsistencia": "Tipo_inconsistencia", "grupo": "Grupo_clustering",
    "rec_inchi": "InChI_recomendado", "rec_confianza": "Confianza", "rec_motivo": "Motivo",
    "rec_fuentes_a_favor": "Fuentes_a_favor", "rec_prioridad_revision": "Prioridad_revision"})
COLS = ["Modelo", "Metabolito", "Nombre", "Compartimentos", "Formula_GEM", "Carga_GEM", "Fuentes", "N_fuentes",
        "N_InChIs_distintos"] + FUENTES + ["Consenso_InChI_completo", "Consenso_estructura", "Tanimoto_minimo",
        "Tipo_inconsistencia", "Grupo_clustering", "Estado_anotacion", "InChI_recomendado", "Confianza",
        "Fuentes_a_favor", "Motivo", "Prioridad_revision"]
if "inchi_features" in globals():   # variables de la Fase 4 del notebook, como referencia
    _f4 = inchi_features[["Modelo", "ID", "SourceSupport", "EstructuraPrincipalConsensus",
                          "InChICompletoConsensus", "StereoConsensus"]].rename(
        columns=lambda c: c if c in ("Modelo", "ID") else f"Fase4_{c}")
    final = final.merge(_f4, on=["Modelo", "ID"], how="left")
    COLS += [c for c in _f4.columns if c.startswith("Fase4_")]
final = final[COLS]
correctos = final[final["Estado_anotacion"] == "Correctamente anotado"]
inconsistentes = final[final["Estado_anotacion"].str.startswith("Inconsistente")]
print(f"Correctamente anotados: {len(correctos)} | Con inconsistencias: {len(inconsistentes)}")
print(inconsistentes["Estado_anotacion"].value_counts().to_string())
prioritarios = final[final["Prioridad_revision"] == "alta"].sort_values(["Tanimoto_minimo", "Modelo"])

resumen_estado = pd.crosstab(final["Modelo"], final["Estado_anotacion"], margins=True, margins_name="Total")
resumen_grupo = pd.crosstab(final["Grupo_clustering"], final["Modelo"], margins=True, margins_name="Total")
resumen_conf = pd.crosstab(final["Modelo"], final["Confianza"], margins=True, margins_name="Total")
print(resumen_estado.to_string()); print(); print(resumen_grupo.to_string()); print(); print(resumen_conf.to_string())

### 10. Excel de entrega

In [ ]:
LEEME = pd.DataFrame([
    ("Resumen_estado", "Cuántos metabolitos por modelo están bien anotados, inconsistentes o sin evidencia."),
    ("Resumen_grupos", "Cuántos metabolitos de cada modelo caen en cada grupo."),
    ("Resumen_confianza", "Nivel de confianza de la recomendación por modelo."),
    ("Analisis_general", "Fase 6.1: consistentes vs inconsistentes y cuántos tienen cada tipo de inconsistencia."),
    ("Discrepancia_fuente", "Qué fuente se aparta de la estructura ganadora (metabolitos con ≥2 bases externas), en %."),
    ("Seleccion_k", "Silhouette y Davies-Bouldin para K-means y Ward, k = 2..8."),
    ("Perfil_grupos", "Promedio de cada variable por grupo (base de la interpretación química)."),
    ("Tabla_final", "Fase 7: una fila por metabolito (compartimentos fusionados)."),
    ("Correctamente_anotados", "Lista: el GEM coincide con el consenso externo y su carga/fórmula con la estructura."),
    ("Con_inconsistencias", "Lista: todos los que tienen algún Estado_anotacion 'Inconsistente: …'."),
    ("Casos_prioritarios", "Lista: la estructura del GEM difiere del consenso de ≥2 bases externas."),
    ("Efectos_en_Fase4", "Cuántas diferencias de 'InChI completo' de la Fase 4 son notación, protonación o estructura."),
    ("", ""),
    ("Consenso_InChI_completo", "Fracción de fuentes cuyo InChI completo (texto) es igual al más frecuente."),
    ("Consenso_estructura", "Fracción de fuentes compatibles con la estructura ganadora: misma constitución neutralizada y "
                            "sin configuraciones R/S o E/Z opuestas (una fuente que no define un centro sí es compatible)."),
    ("Tanimoto_minimo", "Similitud mínima entre pares de fuentes (Morgan r=2) sobre las estructuras neutralizadas con RDKit."),
    ("Tipo_inconsistencia", "formula, conectividad, anillo_cadena, hidrogenos, estereo_contradictoria, estereo_incompleta, carga. "
                            "Describe a todas las fuentes, aunque solo una minoritaria discrepe."),
    ("Estado_anotacion", "Juzga al GEM: 'Correctamente anotado' si coincide con el consenso externo y su carga/fórmula con la "
                         "estructura, aunque alguna fuente minoritaria discrepe (ver Tipo_inconsistencia)."),
    ("Confianza", "alta: todas compatibles y ≥2 bases externas; media: mayoría con ≥1 base externa; revision_manual: no se puede "
                  "decidir (Estado_anotacion dice si es por conflicto o por falta de bases externas); sin_datos: sin InChI."),
    ("Prioridad_revision", "alta: el GEM difiere del consenso de ≥2 bases externas; media: fuentes en conflicto; baja: el resto."),
    ("Fase4_*", "Variables de la Fase 4 (inchi_features) del primer compartimento, como referencia."),
], columns=["Hoja / columna", "Qué es"])

ruta = os.path.join(SALIDA, "tabla_final_fase7.xlsx")
with pd.ExcelWriter(ruta) as xw:
    LEEME.to_excel(xw, "Leeme", index=False)
    resumen_estado.to_excel(xw, "Resumen_estado")
    resumen_grupo.to_excel(xw, "Resumen_grupos")
    resumen_conf.to_excel(xw, "Resumen_confianza")
    conteo.merge(general, on="Modelo", how="right").to_excel(xw, "Analisis_general", index=False)
    discrepa.to_excel(xw, "Discrepancia_fuente")
    sel.round(3).to_excel(xw, "Seleccion_k", index=False)
    perfil.round(3).to_excel(xw, "Perfil_grupos", index=False)
    final.to_excel(xw, "Tabla_final", index=False)
    correctos.to_excel(xw, "Correctamente_anotados", index=False)
    inconsistentes.to_excel(xw, "Con_inconsistencias", index=False)
    prioritarios.to_excel(xw, "Casos_prioritarios", index=False)
    if "tabla_efectos" in globals():
        tabla_efectos.to_excel(xw, "Efectos_en_Fase4")
print("Guardado:", ruta)